In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 30 — RAG Evaluation Dashboard Notebook
## Compare Chunking, Retrieval, and Groundedness Across Strategies

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup Evaluation Framework

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Source document
source_text = """
Kubernetes is a container orchestration platform. It manages containerized
applications across multiple hosts. Key components include pods (smallest unit),
services (stable networking), deployments (declarative updates), and ingress
(external access). The control plane runs the API server, scheduler, and etcd.
Worker nodes run kubelet and container runtime. Horizontal Pod Autoscaler (HPA)
scales pods based on CPU/memory metrics. Kubernetes supports rolling updates
for zero-downtime deployments. Namespaces provide logical isolation. ConfigMaps
and Secrets manage configuration. Persistent Volumes handle storage needs.
"""

# Ground truth QA pairs
eval_set = [
    {"q": "What is the smallest unit in Kubernetes?", "a": "Pods"},
    {"q": "What provides stable networking?", "a": "Services"},
    {"q": "What does HPA scale based on?", "a": "CPU/memory metrics"},
    {"q": "How does Kubernetes handle zero-downtime updates?", "a": "Rolling updates"},
    {"q": "What manages configuration?", "a": "ConfigMaps and Secrets"},
]
print(f"Source: {len(source_text)} chars, Eval set: {len(eval_set)} QA pairs")

## Step 2 — Compare Chunking Strategies

In [ ]:
chunk_configs = [
    {"name": "small", "chunk_size": 100, "overlap": 10},
    {"name": "medium", "chunk_size": 250, "overlap": 30},
    {"name": "large", "chunk_size": 500, "overlap": 50},
]

doc = Document(page_content=source_text, metadata={"source": "k8s_doc"})
results = {}

for config in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"], chunk_overlap=config["overlap"])
    chunks = splitter.split_documents([doc])

    vs = Chroma.from_documents(chunks, embeddings,
        collection_name=f"eval_{config['name']}")

    qa_prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="Answer using the context only.\nContext: {context}\nQ: {question}\nA:")

    qa = RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff",
        retriever=vs.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": qa_prompt},
    )

    config_results = []
    for item in eval_set:
        result = qa.invoke({"query": item["q"]})
        answer = result["result"]
        gt = item["a"]
        # Simple contains check for evaluation
        correct = gt.lower() in answer.lower()
        config_results.append({
            "question": item["q"],
            "expected": gt,
            "got": answer[:100],
            "correct": correct,
            "n_sources": len(result["source_documents"]),
        })

    accuracy = sum(r["correct"] for r in config_results) / len(config_results)
    results[config["name"]] = {"accuracy": accuracy, "n_chunks": len(chunks), "details": config_results}
    print(f"  {config['name']}: {len(chunks)} chunks, accuracy={accuracy:.0%}")

## Step 3 — Dashboard Summary

In [ ]:
print("\n" + "="*60)
print("RAG EVALUATION DASHBOARD")
print("="*60)

print(f"\n{'Strategy':<12} {'Chunks':>8} {'Accuracy':>10} {'Status':>10}")
print("-"*45)
for name, data in results.items():
    status = "✓ GOOD" if data["accuracy"] >= 0.8 else "⚠ CHECK" if data["accuracy"] >= 0.6 else "✗ POOR"
    print(f"{name:<12} {data['n_chunks']:>8} {data['accuracy']:>9.0%} {status:>10}")

best = max(results.items(), key=lambda x: x[1]["accuracy"])
print(f"\nBest strategy: {best[0]} ({best[1]['accuracy']:.0%} accuracy)")

# Per-question breakdown for best strategy
print(f"\nDetailed results for '{best[0]}':")
for r in best[1]["details"]:
    icon = "✓" if r["correct"] else "✗"
    print(f"  {icon} Q: {r['question'][:40]}...")
    print(f"    Expected: {r['expected']} | Got: {r['got'][:60]}")

## Step 4 — Groundedness Evaluation

In [ ]:
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rate if the answer is grounded in the context. "
     "Return a score from 0-10 and one word: 'grounded' or 'hallucinated'."),
    ("human", "Context: {context}\nAnswer: {answer}\nScore and verdict:")
])
judge_chain = judge_prompt | llm | StrOutputParser()

best_strategy = best[0]
best_results = best[1]["details"]

print("Groundedness Check:")
for r in best_results:
    verdict = judge_chain.invoke({"context": source_text, "answer": r["got"]})
    print(f"  Q: {r['question'][:40]}... → {verdict.strip()[:30]}")

## What You Learned
- **Systematic evaluation** of RAG configurations
- **Chunking strategy comparison** with ground truth
- **Groundedness scoring** of generated answers
- **Dashboard-style reporting** for RAG quality metrics